In [ ]:
from bigmodule import M
import bigtrader

# <aistudiograph>

# @param(id="m4", name="initialize")
def m4_initialize_bigquant_run(context):
    import pandas as pd
    

    # 手续费（默认示例配置）
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))

    # ====== 1) 读取 Excel 股票清单 ======
    # 你需要把文件放到策略可访问路径；请修改为你的实际路径
    context.excel_path = "/home/jovyan/work/stock_list.xlsx"
    # Excel 里股票代码列名：请按你的文件修改
    context.excel_code_col = "code"

    df = pd.read_excel(context.excel_path)

    if context.excel_code_col not in df.columns:
        raise ValueError(f"Excel中找不到列: {context.excel_code_col}, 实际列: {list(df.columns)}")

    raw_codes = df[context.excel_code_col].dropna().astype(str).str.strip().tolist()

    def normalize_a_share_code(x: str):
        # 兼容: 600000 / 600000.SH / SH600000 / 000001.SZ 等
        x = x.upper().replace(" ", "")
        x = x.replace("SH", "").replace("SZ", "").replace(".", "")
        # 现在期望是 6位数字
        digits = "".join([c for c in x if c.isdigit()])
        if len(digits) != 6:
            return None
        if digits.startswith(("60", "68")):
            return digits + ".SH"
        else:
            return digits + ".SZ"

    instruments = []
    for c in raw_codes:
        nc = normalize_a_share_code(c)
        if nc is not None:
            instruments.append(nc)

    instruments = sorted(list(set(instruments)))
    if len(instruments) == 0:
        raise ValueError("Excel 解析后股票列表为空，请检查文件内容/列名/格式")

    context.watch_instruments = instruments

    # ====== 交易参数 ======
    context.max_single_position = 0.05   # 单票最大仓位(可改)
    context.total_target_position = 0.95 # 总目标仓位上限(可改)

    # 记录买入时间，用于区分 5日内/5日后逻辑
    context.buy_dt = {}  # {instrument: datetime}

# @param(id="m4", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m4_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m4", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m4_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m4", name="handle_data")
def m4_handle_data_bigquant_run(context, data):
    import pandas as pd

    # 取当前分钟/日
    current_dt = data.current_dt
    current_date = current_dt.strftime("%Y-%m-%d")

    # 从离线特征表中取“当日”的日线 close/ma5/ma60
    day_df = context.data[context.data["date"] == current_date]
    if day_df is None or len(day_df) == 0:
        return

    day_df = day_df[day_df["instrument"].isin(context.watch_instruments)]
    if len(day_df) == 0:
        return

    # 当前持仓
    current_hold_instruments = set(context.get_account_positions().keys())

    # ========== 卖出逻辑（优先执行）==========
    for ins in list(current_hold_instruments):
        if ins not in context.watch_instruments:
            continue

        row = day_df[day_df["instrument"] == ins]
        if len(row) == 0:
            continue

        close_d = float(row["close"].iloc[0])
        ma5 = float(row["ma5"].iloc[0])
        ma60 = float(row["ma60"].iloc[0])

        # 计算已持有的交易日数：用自然日近似（更严谨需交易日历，这里用回测日期差近似）
        buy_dt = context.buy_dt.get(ins, None)
        hold_days = None
        if buy_dt is not None:
            hold_days = (current_dt.date() - buy_dt.date()).days
        else:
            hold_days = 9999

        # 0~5日：跌破 MA60 卖出
        if hold_days <= 5:
            if close_d < ma60:
                context.order_target_percent(ins, 0)
                context.buy_dt.pop(ins, None)
                continue

        # >5日：跌破 MA5 卖出（与你“没有超出5日均线/跌破5日均线”合并实现）
        if hold_days > 5:
            if close_d < ma5:
                context.order_target_percent(ins, 0)
                context.buy_dt.pop(ins, None)
                continue

    # ========== 买入逻辑：盘中从下向上穿越“日线MA60”==========
    # 候选条件：最新日线收盘价 < 日线MA60
    candidates = day_df[day_df["close"] < day_df["ma60"]]["instrument"].tolist()
    if len(candidates) == 0:
        return

    # 为了判断“由下向上穿越”，用前一分钟价格与当前价格相对 MA60 的关系判断
    # 取最近2个bar的分钟收盘价（若取不到则跳过）
    for ins in candidates:
        if ins in current_hold_instruments:
            continue

        row = day_df[day_df["instrument"] == ins]
        if len(row) == 0:
            continue
        ma60 = float(row["ma60"].iloc[0])

        try:
            # BigTrader 分钟回测里常用 history；若你的环境不支持，请告知我改成可用API
            h = data.history(ins, ["close"], 2, "1m")
            if h is None or len(h) < 2:
                continue
            prev_price = float(h["close"].iloc[-2])
            curr_price = float(h["close"].iloc[-1])
        except Exception:
            continue

        # 从下向上穿越 MA60：prev <= ma60 且 curr > ma60
        if (prev_price <= ma60) and (curr_price > ma60):
            # 控制总仓位与单票仓位
            # 简化：按固定单票仓位买入，直到接近总目标仓位
            target_w = min(context.max_single_position, context.total_target_position)

            context.order_target_percent(ins, target_w)
            context.buy_dt[ins] = current_dt

# @param(id="m4", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m4_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m4", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m4_handle_order_bigquant_run(context, order):
    pass

# @param(id="m4", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m4_after_trading_bigquant_run(context, data):
    pass

# @module(position="140,40", comment="""A股-基础选股：作为数据底表（实际交易标的由Excel控制）""")
m1 = M.cn_stock_basic_selector.v8(
    exchanges=["""上交所""", """深交所""", """北交所"""],
    list_sectors=["""主板""", """科创板""", """创业板"""],
    indexes=["""沪深300""", """中证500""", """中证1000""", """上证50""", """上证指数""", """深证成指""", """创业板指""", """科创50""", """科创100""", """中证100"""],
    st_statuses=["""正常""", """ST""", """*ST"""],
    sw2021_industries=["""农林牧渔""", """采掘""", """基础化工""", """钢铁""", """有色金属""", """建筑建材""", """机械设备""", """电子""", """汽车""", """交运设备""", """信息设备""", """家用电器""", """食品饮料""", """纺织服饰""", """轻工制造""", """医药生物""", """公用事业""", """交通运输""", """房地产""", """金融服务""", """商贸零售""", """社会服务""", """信息服务""", """银行""", """非银金融""", """综合""", """建筑材料""", """建筑装饰""", """电力设备""", """国防军工""", """计算机""", """传媒""", """通信""", """煤炭""", """石油石化""", """环保""", """美容护理"""],
    drop_suspended=True,
    m_name="""m1"""
)

# @module(position="140,170", comment="""计算日线 MA5/MA60 与 close，供盘中逻辑使用""")
m2 = M.input_features_dai.v30(
    input_2=m1.data,
    mode="""表达式""",
    expr="""close AS close
m_avg(close, 5) AS ma5
m_avg(close, 60) AS ma60""",
    expr_filters="""1=1""",
    expr_tables="""cn_stock_prefactors""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据, 构建因子等, 如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B
-- 使用数据输入1/2/3里的字段: e.g. input_1.close, input_1.* EXCLUDE(date, instrument)

SELECT
    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    m_lag(close, 90) / close AS return_90,
    m_lag(close, 30) / close AS return_30,
    -- 下划线开始命名的列是中间变量, 不会在最终结果输出 (e.g. _rank_return_90)
    c_pct_rank(-return_90) AS _rank_return_90,
    c_pct_rank(return_30) AS _rank_return_30,

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_bar1d https://bigquant.com/data/datasources/cn_stock_bar1d
    cn_stock_prefactors
    -- SQL 模式不会自动join输入数据源, 可以根据需要自由灵活的使用
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤, 在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤, 在窗口等计算算子之后执行, 比如 m_lag(close, 3) AS close_3, 对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
    -- _rank_return_90 是窗口函数结果，需要放在 QUALIFY 里
    AND _rank_return_90 > 0.1
    AND _rank_return_30 < 0.1
-- 按日期和股票代码排序, 从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m2"""
)

# @module(position="140,300", comment="""抽取回测数据（包含计算均线所需的前置天数）""")
m3 = M.extract_data_dai.v20(
    sql=m2.data,
    start_date="""2022-01-01""",
    start_date_bound_to_trading_date=True,
    end_date="""2024-12-31""",
    end_date_bound_to_trading_date=True,
    before_start_days=120,
    keep_before=False,
    debug=False,
    m_name="""m3"""
)

# @module(position="140,430", comment="""回测：minute 频率以支持盘中穿越监控""")
m4 = M.bigtrader.v43(
    data=m3.data,
    start_date="""""",
    end_date="""""",
    initialize=m4_initialize_bigquant_run,
    before_trading_start=m4_before_trading_start_bigquant_run,
    handle_tick=m4_handle_tick_bigquant_run,
    handle_data=m4_handle_data_bigquant_run,
    handle_trade=m4_handle_trade_bigquant_run,
    handle_order=m4_handle_order_bigquant_run,
    after_trading=m4_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""minute""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""1""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""close""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="""m4"""
)
# </aistudiograph>